# 00 · Tổng quan pipeline và cấu hình chung

Notebook này là bản đồ của cả dự án. Toàn bộ pipeline có các bước:

| Bước | Notebook | File code chính | Việc làm |
|---|---|---|---|
| 1 | `01_tien_xu_ly_du_lieu` | `tcfall/data.py`, `pipeline/s01_audit_data.py` | Nạp nhãn, gán split, làm sạch, kiểm toán dữ liệu |
| 2 | `02_lay_mau_video_va_trich_dac_trung` | `tcfall/video.py`, `tcfall/features.py`, `tcfall/preprocess_gpu.py` | Lấy mẫu 7.5 fps, trích feature VideoMAE |
| 3 | `03_cat_doan_va_cache_logits` | `tcfall/segments.py`, `pipeline/s03_cache_logits.py` | Cắt 18 feature cho mỗi segment, chạy classifier, lưu logits |
| 4 | `04_backbone_mo_hinh` | `FeatureTransformer`, `tcfall/baseline.py` | Kiến trúc classifier trên feature |
| 5 | `05_finetune_itw` | `pipeline/s05b_finetune_itw.py` | Fine-tune classifier trên ItW-train |
| 6 | `06_luat_chuyen_trang_thai` | `tcfall/conditioners.py`, `src/conditioning.py` | Rule, HMM, prior shift, Hybrid |
| 7 | `07_danh_gia_streaming` | `tcfall/metrics.py`, `pipeline/s07_stream_eval.py` | Metric, kiểm định, đánh giá streaming |
| 8 | `08_realtime_engine` | `tcfall/realtime.py` | Engine suy luận thời gian thực cho app |

Mọi cell code trong các notebook được **chép nguyên văn** từ file nguồn trong repo; cell markdown phía trên giải thích đoạn đó làm gì và vì sao.
Các cell định nghĩa hàm chạy được ngay; các cell `main()` cần dữ liệu (thư mục `data/`, `outputs_v2/`) nên chỉ để đọc hiểu, trừ khi bạn đã tải dữ liệu theo README.

Một số cell có tiêu đề **Ví dụ nhỏ**: đó là code minh hoạ thêm, chạy được không cần dữ liệu.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# Notebook nằm trong thư mục notebooks/ -> thêm thư mục gốc repo vào sys.path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)

Repo root: <repo>


**`tcfall/config.py` · phần import và hằng số**

File cấu hình dùng chung. Đáng chú ý:
- `ROOT, DATA, OUTPUTS, CKPT_DIR`: đường dẫn; mọi kết quả của pipeline mới nằm trong `outputs_v2/`.
- `VIDEOMAE_NAME`: backbone video VideoMAE-Base đã fine-tune trên Kinetics-400.
- `TARGET_FPS = 7.5`, `WINDOW_FRAMES = 16`: mỗi cửa sổ VideoMAE gồm 16 frame ≈ 2.13 giây.
- `NUM_FEATURES = 18`: classifier nhìn 18 feature liên tiếp (≈ 2.27 s ngữ cảnh).
- `CLASS_NAMES`: 10 lớp lõi của OmniFall (bộ đầy đủ có 16 lớp, nhóm dùng 10).
- `TRANSIENT_TO_STATIC`: hành động chuyển tiếp → trạng thái tĩnh theo sau (fall→fallen, sit_down→sitting, lie_down→lying, stand_up→standing). Đây là cấu trúc mà luật chuyển trạng thái khai thác.
- `FEATURE_VARIANTS`: `legacy` (cách lấy mẫu cũ, gây lỗi trôi) và `timefix` (lấy mẫu theo thời gian thật).

In [2]:
from __future__ import annotations
from pathlib import Path
ROOT = Path(ROOT / "tcfall/config.py").resolve().parent.parent
DATA = ROOT / "data" / "omnifall"
EXPERIMENTS_ROOT = ROOT / "omnifall-experiments"
OUTPUTS = ROOT / "outputs_v2"          # mọi kết quả mới của pipeline v2 nằm ở đây
CKPT_DIR = ROOT / "checkpoints"
LABELS_RAW = DATA / "labels" / "dataset"
SPLITS = DATA / "splits"
OOPS_VIDEO = DATA / "OOPS" / "video"
VIDEOMAE_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"
TARGET_FPS = 7.5
WINDOW_FRAMES = 16                     # 1 cửa sổ VideoMAE = 16 frame = 2.13 s
FEATURE_STRIDE = 2 / 15                # 1 frame ở 7.5 fps
NUM_FEATURES = 18                      # classifier nhận 18 feature liên tiếp
FEATURE_DIM = 768
BASELINE_CKPT = (EXPERIMENTS_ROOT / "outputs" / "2026-09-08" / "21-55-48" /
                 "feature_transformer_768_2l" / "checkpoint-7000" / "model.safetensors")
CLASS_NAMES = ["walk", "fall", "fallen", "sit_down", "sitting",
               "lie_down", "lying", "stand_up", "standing", "other"]
NUM_CLASSES = len(CLASS_NAMES)
FALL, FALLEN, LIE_DOWN, LYING = 1, 2, 5, 6
TRANSIENT_TO_STATIC = {1: 2, 3: 4, 5: 6, 7: 8}
LABEL_FILES = {  # tên dataset -> file nhãn gốc
    "of-syn": "of-syn.csv", "gmdcsa24": "GMDCSA24.csv", "le2i": "le2i.csv",
    "caucafall": "caucafall.csv", "OOPS": "OOPS.csv",
}
FEATURE_VARIANTS = ("legacy", "timefix")

**`tcfall/config.py` · feature_dir**

Trả về thư mục chứa feature của một bộ dữ liệu theo biến thể lấy mẫu.

In [3]:
def feature_dir(dataset: str, variant: str) -> Path:
    return DATA / dataset / "features_v2" / variant

**Ví dụ nhỏ:** in ra các lớp và ánh xạ chuyển tiếp → tĩnh.

In [4]:
from tcfall.config import CLASS_NAMES, TRANSIENT_TO_STATIC, WINDOW_FRAMES, TARGET_FPS
for i, c in enumerate(CLASS_NAMES):
    print(i, c)
print({CLASS_NAMES[a]: CLASS_NAMES[b] for a, b in TRANSIENT_TO_STATIC.items()})
print("Độ dài 1 cửa sổ VideoMAE:", WINDOW_FRAMES / TARGET_FPS, "giây")

0 walk
1 fall
2 fallen
3 sit_down
4 sitting
5 lie_down
6 lying
7 stand_up
8 standing
9 other
{'fall': 'fallen', 'sit_down': 'sitting', 'lie_down': 'lying', 'stand_up': 'standing'}
Độ dài 1 cửa sổ VideoMAE: 2.1333333333333333 giây


## ▶ Chạy thật trên dữ liệu thật

Phần trên định nghĩa code. Từ đây là các ô chạy thật bằng dữ liệu, checkpoint và GPU của dự án.

Đọc cấu hình thật đang được dùng và kiểm tra những gì có sẵn trên máy.

In [5]:
import json
from pathlib import Path
from tcfall import config as C

best = json.loads((C.OUTPUTS / "best_config.json").read_text(encoding="utf-8"))["stream_ft"]
print("Lớp:", dict(enumerate(C.CLASS_NAMES)))
print(f"Cửa sổ: {C.WINDOW_FRAMES} ảnh ở {C.TARGET_FPS} fps = {C.WINDOW_FRAMES / C.TARGET_FPS:.2f} s; {C.NUM_FEATURES} vector, mỗi vector {C.FEATURE_DIM} chiều")
print("Cấu hình realtime đã chọn trên val:", {k: best[k] for k in ("name", "input", "mode", "segment_hop", "tau", "window")})
checks = {"Nhãn": C.LABELS_RAW, "Split": C.SPLITS, "Video OOPS": C.OOPS_VIDEO, "Feature timefix": C.feature_dir("OOPS", "timefix"),
          "Checkpoint gốc": C.BASELINE_CKPT, "Checkpoint fine-tune": C.OUTPUTS / "finetune" / "baseline_ft_itw_seed42.pt"}
for name, p in checks.items():
    n = f"{sum(1 for _ in p.glob('*'))} mục" if p.is_dir() else ("có" if p.exists() else "")
    print(f"  {name:22s} {'CÓ ' + n if p.exists() else 'THIẾU'}")

Lớp: {0: 'walk', 1: 'fall', 2: 'fallen', 3: 'sit_down', 4: 'sitting', 5: 'lie_down', 6: 'lying', 7: 'stand_up', 8: 'standing', 9: 'other'}
Cửa sổ: 16 ảnh ở 7.5 fps = 2.13 s; 18 vector, mỗi vector 768 chiều
Cấu hình realtime đã chọn trên val: {'name': 'rule τ=1,W=5,hop=0.5', 'input': 'spatial', 'mode': 'rule', 'segment_hop': 0.5, 'tau': 1, 'window': 5}
  Nhãn                   CÓ 5 mục
  Split                  CÓ 3 mục
  Video OOPS             CÓ 1 mục
  Feature timefix        CÓ 818 mục
  Checkpoint gốc         CÓ có
  Checkpoint fine-tune   CÓ có
